In [3]:
import re
import pandas as pd
import numpy as np
from advanced_ta import LorentzianClassification as LC

def _to_pd_rule(tf: str) -> str:
    """
    Convert timeframe like '5m','15min','1h','1D' to pandas rule ('5min','15min','1h','1d').
    """
    s = tf.strip().lower()
    m = re.fullmatch(r'(\d+)\s*([a-z]+)', s)
    if not m:
        raise ValueError(f"Unrecognized timeframe: {tf}")
    n, u = m.groups()
    u = {'m':'min','min':'min','mins':'min',
         'h':'h','hr':'h','hrs':'h','hour':'h','hours':'h',
         'd':'d','day':'d','days':'d'}[u]
    return f"{int(n)}{u}"

def _prep_prices(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Lowercase OHLCV columns and set a tz-aware DatetimeIndex.
    Requires columns for ['open','high','low','close','volume'] and either 'ts' column or DatetimeIndex.
    """
    df = df_raw.copy()

    # Ensure datetime index
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], utc=True, errors='coerce')
        df = df.dropna(subset=['ts']).set_index('ts')
    elif not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True, errors='coerce')
        df = df[~df.index.isna()]

    # Standardize columns
    df = df.rename(columns=str.lower)
    needed = ['open','high','low','close','volume']
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns: {missing}")

    # Enforce numeric
    df[needed] = df[needed].apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=needed).sort_index()
    return df

def _resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    ohlc = {
        'open':  'first',
        'high':  'max',
        'low':   'min',
        'close': 'last',
        'volume':'sum'
    }
    out = (df.resample(rule, label='right', closed='right')
             .agg(ohlc)
             .dropna())
    return out

def _find_col(df: pd.DataFrame, names) -> str | None:
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n in df.columns: return n
        if n.lower() in m:  return m[n.lower()]
    return None


import pandas as pd
import numpy as np
from advanced_ta import LorentzianClassification as LC

def run_lorentzian_signals(
    df_raw: pd.DataFrame,
    tf: str = "5m",
    start: str | None = None,
    end: str | None = None,
    *,
    neighbors_count: int = 8,
    max_bars_back: int | None = None,
    use_dynamic_exits: bool = False,
    use_volatility_filter: bool = True,
    use_regime_filter: bool = True,
    use_adx_filter: bool = False,
    regime_threshold: float = -0.1,
    adx_threshold: int = 20,
    color_compression: float = 1.0,
    strong_threshold: float = 2/3,     # ≥ this ⇒ "strong", else "mild"
    features: list | None = None
):
    # --- Prep / index ---
    df = df_raw.copy()
    if "ts" in df.columns:
        df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
        df = df.dropna(subset=["ts"]).set_index("ts")
    elif not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True, errors="coerce")
    df = df[~df.index.isna()].sort_index()
    df = df.rename(columns=str.lower)[["open","high","low","close","volume"]].astype(float)

    # --- slice window ---
    if start: df = df[df.index >= pd.to_datetime(start, utc=True)]
    if end:   df = df[df.index <= pd.to_datetime(end,   utc=True)]

    # --- resample ---
    TF_MAP = {"5m":"5min","15m":"15min","30m":"30min","1h":"1h","2h":"2h","4h":"4h","1d":"1D"}
    rule = TF_MAP.get(tf.lower(), tf)
    if rule:
        agg = {"open":"first","high":"max","low":"min","close":"last","volume":"sum"}
        df = df.resample(rule, label="right", closed="right").agg(agg).dropna(how="any")
    if df.empty:
        return pd.DataFrame(columns=["timestamp","event","price_close","prediction","strength","grade"]), df, None

    # --- features (Pine-ish defaults) ---
    if features is None:
        features = [
            LC.Feature("RSI", 14, 2),
            LC.Feature("WT",  10, 11),
            LC.Feature("CCI", 20, 1),
            LC.Feature("ADX", 20, 2),
            LC.Feature("RSI", 9,  1),
        ]

    # --- LC settings/filters ---
    max_bars_back = max_bars_back or len(df)
    settings = LC.Settings(
        source=df["close"], neighborsCount=neighbors_count,
        maxBarsBack=max_bars_back, useDynamicExits=use_dynamic_exits
    )
    filterSettings = LC.FilterSettings(
        useVolatilityFilter=use_volatility_filter,
        useRegimeFilter=use_regime_filter,
        useAdxFilter=use_adx_filter,
        regimeThreshold=regime_threshold,
        adxThreshold=adx_threshold
    )

    # --- run LC ---
    lc = LC(df[["open","high","low","close","volume"]], features=features,
            settings=settings, filterSettings=filterSettings)

    # pull working frame (fallback via dump)
    out = None
    for attr in ("df","data","result","_df"):
        cand = getattr(lc, attr, None)
        if isinstance(cand, pd.DataFrame):
            out = cand; break
    if out is None:
        import tempfile, os
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".csv"); tmp.close()
        try:
            lc.dump(tmp.name); out = pd.read_csv(tmp.name)
        finally:
            try: os.remove(tmp.name)
            except Exception: pass

    # --- normalize time index on output ---
    out = out.copy()
    if "ts" in out.columns:
        out["ts"] = pd.to_datetime(out["ts"], utc=True, errors="coerce")
        out = out.dropna(subset=["ts"]).set_index("ts")
    elif not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index, utc=True, errors="coerce")
    out = out[~out.index.isna()].sort_index()

    # --- helpers ---
    def find_col(dfX, names):
        m = {c.lower(): c for c in dfX.columns}
        for n in names:
            if n in dfX.columns: return n
            if n.lower() in m:   return m[n.lower()]
        return None

    long_col  = find_col(out, ["startLongTrade","start_long_trade","buy_signal","buy"])
    short_col = find_col(out, ["startShortTrade","start_short_trade","sell_signal","sell"])
    close_col = find_col(out, ["close"]) or "close"
    pred_col  = find_col(out, ["prediction","signal","label","y_pred","pred"])

    # --- prediction → numeric & strength/grade ---
    pred = pd.to_numeric(out[pred_col], errors="coerce").fillna(0.0) if pred_col else pd.Series(0.0, index=out.index)
    denom = max(1e-9, (neighbors_count / max(1e-9, color_compression)))
    strength = (pred.abs() / denom).clip(0, 1)
    grade = np.where(strength >= float(strong_threshold), "strong", "mild")

    # --- build BUY/SELL only ---
    events = []

    if long_col is not None:
        m = pd.to_numeric(out[long_col], errors="coerce").fillna(0) > 0
        if m.any():
            events.append(pd.DataFrame({
                "timestamp": out.index[m],
                "event": "BUY",
                "price_close": pd.to_numeric(out.loc[m, close_col], errors="coerce"),
                "prediction": pred.loc[m].values,
                "strength": strength.loc[m].values,
                "grade": grade[m],
            }))

    if short_col is not None:
        m = pd.to_numeric(out[short_col], errors="coerce").fillna(0) > 0
        if m.any():
            events.append(pd.DataFrame({
                "timestamp": out.index[m],
                "event": "SELL",
                "price_close": pd.to_numeric(out.loc[m, close_col], errors="coerce"),
                "prediction": pred.loc[m].values,
                "strength": strength.loc[m].values,
                "grade": grade[m],
            }))

    # Fallback to sign(pred) if no explicit start columns
    if not events and pred_col is not None:
        s = np.sign(pred).astype(int).clip(-1, 1)
        m_buy, m_sell = s > 0, s < 0
        if m_buy.any():
            events.append(pd.DataFrame({
                "timestamp": out.index[m_buy],
                "event": "BUY",
                "price_close": pd.to_numeric(out.loc[m_buy, close_col], errors="coerce"),
                "prediction": pred.loc[m_buy].values,
                "strength": strength.loc[m_buy].values,
                "grade": np.where(strength.loc[m_buy] >= float(strong_threshold), "strong", "mild"),
            }))
        if m_sell.any():
            events.append(pd.DataFrame({
                "timestamp": out.index[m_sell],
                "event": "SELL",
                "price_close": pd.to_numeric(out.loc[m_sell, close_col], errors="coerce"),
                "prediction": pred.loc[m_sell].values,
                "strength": strength.loc[m_sell].values,
                "grade": np.where(strength.loc[m_sell] >= float(strong_threshold), "strong", "mild"),
            }))

    signals = (pd.concat(events, ignore_index=True) if events else
               pd.DataFrame(columns=["timestamp","event","price_close","prediction","strength","grade"]))
    signals = signals.sort_values("timestamp").reset_index(drop=True)

    return signals, out, lc

In [16]:
import pandas as pd
import numpy as np

def compute_pnl_from_signals(
    signals: pd.DataFrame,
    *,
    price_col: str = "price_close",
    event_col: str = "event",            # expects "BUY" / "SELL" (case-insensitive)
    time_col: str = "timestamp",
    roundtrip_fee: float = 0.002,        # 0.2% roundtrip
    initial_equity: float = 1.0,
    close_on_last: bool = False,         # if True, force-close any open position at the last signal price
) -> tuple[pd.DataFrame, pd.Series, dict]:
    """
    Flips between long (+1) and short (-1) on BUY/SELL signals.
    Executes at the signal's price on the signal's timestamp.
    Fees: half at entry and half at exit (flip charges both on one bar).

    Returns:
        trades_df: one row per CLOSED trade with gross/net returns and cumulative equity
        equity: stepwise equity Series indexed by trade exit time
        summary: dict with totals (trades, winrate, total net return, etc.)
    """
    if signals is None or signals.empty:
        equity = pd.Series([initial_equity], index=pd.Index([], dtype="datetime64[ns, UTC]"), name="equity")
        return pd.DataFrame(), equity, {"trades": 0, "net_return": 0.0, "final_equity": initial_equity}

    df = signals.copy()

    # Normalize columns
    if time_col not in df.columns:
        raise ValueError(f"signals missing '{time_col}' column")
    if price_col not in df.columns:
        raise ValueError(f"signals missing '{price_col}' column")
    if event_col not in df.columns:
        raise ValueError(f"signals missing '{event_col}' column")

    # Time/index hygiene
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce")
    df = df.dropna(subset=[time_col, price_col])
    df = df.sort_values(time_col).reset_index(drop=True)

    # Canonicalize events and drop unknowns
    df[event_col] = df[event_col].astype(str).str.upper().str.strip()
    df = df[df[event_col].isin(["BUY", "SELL"])].copy()
    if df.empty:
        equity = pd.Series([initial_equity], index=pd.Index([], dtype="datetime64[ns, UTC]"), name="equity")
        return pd.DataFrame(), equity, {"trades": 0, "net_return": 0.0, "final_equity": initial_equity}

    # Remove consecutive duplicates (BUY after BUY, SELL after SELL)
    dedup_mask = df[event_col].ne(df[event_col].shift())
    df = df[dedup_mask].reset_index(drop=True)

    half_fee = roundtrip_fee * 0.5

    # State
    pos = 0                     # 0=flat, +1=long, -1=short
    entry_price = None
    entry_time  = None
    equity_val = initial_equity

    records = []
    equity_points = []          # (exit_time, equity_val) after each closed trade

    def close_trade(exit_time, exit_price, new_pos_after):
        nonlocal pos, entry_price, entry_time, equity_val

        if pos == 0 or entry_price is None:
            return

        # Gross return
        if pos == +1:
            gross = (exit_price / entry_price) - 1.0
        else:  # pos == -1
            gross = (entry_price / exit_price) - 1.0

        # Fees:
        # - pay exit half always
        # - if flipping (new_pos_after != 0), also pay entry half for the new position now
        fee_cost = half_fee + (half_fee if new_pos_after != 0 else 0.0)

        net = gross - fee_cost
        equity_val *= (1.0 + net)

        records.append({
            "entry_time": entry_time,
            "exit_time": exit_time,
            "side": "long" if pos == +1 else "short",
            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_return": gross,
            "fee_cost": fee_cost,
            "net_return": net,
            "equity_after": equity_val,
        })
        equity_points.append((exit_time, equity_val))

        # After closing, clear entry; caller will set new entry if needed
        entry_price = None
        entry_time  = None
        pos = 0

    # Walk signals
    for _, row in df.iterrows():
        t = row[time_col]
        p = float(row[price_col])
        evt = row[event_col]

        desired = +1 if evt == "BUY" else -1

        if pos == desired:
            # Same side as current position → ignore
            continue

        # If we have an open trade, close it at this price/time (and pay exit + next entry fee if flipping)
        close_trade(t, p, new_pos_after=desired)

        # Open new trade if desired != 0
        if desired != 0:
            # Opening fee (half roundtrip)
            open_fee = half_fee  # fee is applied to return math at close; we just note entry
            pos = desired
            entry_price = p
            entry_time = t
            # Note: we account for the entry fee when the trade closes (paired with its exit fee).
            # If you prefer to deduct at entry immediately from equity, uncomment next line:
            # equity_val *= (1.0 - open_fee)

    # Optionally force-close the final open trade at the last seen price/time
    if close_on_last and pos != 0 and entry_price is not None:
        last_time = df.iloc[-1][time_col]
        last_price = float(df.iloc[-1][price_col])
        close_trade(last_time, last_price, new_pos_after=0)

    trades_df = pd.DataFrame.from_records(records)
    if not trades_df.empty:
        trades_df = trades_df.sort_values("exit_time").reset_index(drop=True)
        trades_df["cum_return"] = (1.0 + trades_df["net_return"]).cumprod() - 1.0
        equity = pd.Series(
            [initial_equity] + trades_df["equity_after"].tolist(),
            index=pd.DatetimeIndex([trades_df["exit_time"].iloc[0] - pd.Timedelta(nanoseconds=1)] + trades_df["exit_time"].tolist(), tz="UTC"),
            name="equity",
        )
    else:
        equity = pd.Series([initial_equity], index=pd.Index([], dtype="datetime64[ns, UTC]"), name="equity")

    # Summary
    total_net = (trades_df["net_return"] + 1.0).prod() - 1.0 if not trades_df.empty else 0.0
    wins = int((trades_df["net_return"] > 0).sum()) if not trades_df.empty else 0
    losses = int((trades_df["net_return"] <= 0).sum()) if not trades_df.empty else 0
    summary = {
        "trades": len(trades_df),
        "wins": wins,
        "losses": losses,
        "winrate": wins / max(1, (wins + losses)),
        "net_return": total_net,
        "final_equity": equity.iloc[-1] if len(equity) else initial_equity,
    }

    return trades_df, equity, summary


import pandas as pd
import numpy as np
from typing import Tuple, Dict, Optional

# --------------------------------------------------------------------
# Common utilities
# --------------------------------------------------------------------
def _normalize_signals(
    signals: pd.DataFrame,
    *,
    time_col: str = "timestamp",
    price_col: str = "price_close",
    event_col: str = "event",
) -> pd.DataFrame:
    """Sort, sanitize, and standardize BUY/SELL events."""
    if signals is None or signals.empty:
        raise ValueError("signals is empty")

    df = signals.copy()
    if time_col not in df.columns or price_col not in df.columns or event_col not in df.columns:
        raise ValueError(f"signals must have columns: {time_col}, {price_col}, {event_col}")

    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce")
    df = df.dropna(subset=[time_col, price_col]).sort_values(time_col).reset_index(drop=True)
    df[event_col] = df[event_col].astype(str).str.upper().str.strip()
    df = df[df[event_col].isin(["BUY", "SELL"])].copy()
    if df.empty:
        raise ValueError("signals has no BUY/SELL rows after cleaning")

    # remove consecutive duplicates (BUY after BUY, etc.)
    df = df[df[event_col].ne(df[event_col].shift())].reset_index(drop=True)
    return df


def _finish(
    records: list,
    initial_equity: float,
) -> Tuple[pd.DataFrame, pd.Series, Dict]:
    """Build trades_df, equity series, summary from records."""
    trades_df = pd.DataFrame.from_records(records)
    if trades_df.empty:
        equity = pd.Series([initial_equity], name="equity")
        summary = {
            "trades": 0, "wins": 0, "losses": 0,
            "winrate": 0.0, "net_return": 0.0, "final_equity": initial_equity,
        }
        return trades_df, equity, summary

    trades_df = trades_df.sort_values("exit_time").reset_index(drop=True)
    trades_df["cum_return"] = (1.0 + trades_df["net_return"]).cumprod() - 1.0

    equity = pd.Series(
        trades_df["equity_after"].values,
        index=pd.DatetimeIndex(trades_df["exit_time"], tz="UTC"),
        name="equity",
    )

    total_net = (1.0 + trades_df["net_return"]).prod() - 1.0
    wins = int((trades_df["net_return"] > 0).sum())
    losses = int((trades_df["net_return"] <= 0).sum())
    summary = {
        "trades": len(trades_df),
        "wins": wins,
        "losses": losses,
        "winrate": wins / max(1, wins + losses),
        "net_return": total_net,
        "final_equity": float(equity.iloc[-1]),
    }
    return trades_df, equity, summary


# --------------------------------------------------------------------
# Long-only
# --------------------------------------------------------------------
def compute_pnl_long_only(
    signals: pd.DataFrame,
    *,
    price_col: str = "price_close",
    event_col: str = "event",
    time_col: str = "timestamp",
    roundtrip_fee: float = 0.002,   # 0.2% round-trip => 0.1% at entry + 0.1% at exit
    initial_equity: float = 1.0,
    close_on_last: bool = False,
) -> Tuple[pd.DataFrame, pd.Series, Dict]:
    """
    Long-only execution:
      - BUY opens/keeps a long (if already long, ignored)
      - SELL closes a long (if flat, ignored)
      - Never opens short.
    """
    df = _normalize_signals(signals, time_col=time_col, price_col=price_col, event_col=event_col)
    half_fee = roundtrip_fee * 0.5

    pos = 0                 # 0=flat, +1=long
    entry_price = None
    entry_time  = None
    equity_val = initial_equity
    records = []

    def close_long(exit_time, exit_price):
        nonlocal pos, entry_price, entry_time, equity_val
        if pos != +1 or entry_price is None:
            return
        gross = (exit_price / entry_price) - 1.0
        fee_cost = half_fee + half_fee  # entry + exit halves
        net = gross - fee_cost
        equity_val *= (1.0 + net)
        records.append({
            "entry_time": entry_time,
            "exit_time": exit_time,
            "side": "long",
            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_return": gross,
            "fee_cost": fee_cost,
            "net_return": net,
            "equity_after": equity_val,
        })
        pos = 0
        entry_price = None
        entry_time = None

    for _, row in df.iterrows():
        t = row[time_col]
        p = float(row[price_col])
        evt = row[event_col]

        if evt == "BUY":
            if pos == 0:
                # open long (entry fee accounted with exit fee when closing)
                pos = +1
                entry_price = p
                entry_time = t
            # if already long -> ignore
        else:  # SELL
            if pos == +1:
                close_long(t, p)
            # if flat -> ignore

    if close_on_last and pos == +1 and entry_price is not None:
        last_t = df.iloc[-1][time_col]
        last_p = float(df.iloc[-1][price_col])
        close_long(last_t, last_p)

    return _finish(records, initial_equity)


# --------------------------------------------------------------------
# Short-only
# --------------------------------------------------------------------
def compute_pnl_short_only(
    signals: pd.DataFrame,
    *,
    price_col: str = "price_close",
    event_col: str = "event",
    time_col: str = "timestamp",
    roundtrip_fee: float = 0.002,
    initial_equity: float = 1.0,
    close_on_last: bool = False,
) -> Tuple[pd.DataFrame, pd.Series, Dict]:
    """
    Short-only execution:
      - SELL opens/keeps a short (if already short, ignored)
      - BUY closes a short (if flat, ignored)
      - Never opens long.
    """
    df = _normalize_signals(signals, time_col=time_col, price_col=price_col, event_col=event_col)
    half_fee = roundtrip_fee * 0.5

    pos = 0                 # 0=flat, -1=short
    entry_price = None
    entry_time  = None
    equity_val = initial_equity
    records = []

    def close_short(exit_time, exit_price):
        nonlocal pos, entry_price, entry_time, equity_val
        if pos != -1 or entry_price is None:
            return
        gross = (entry_price / exit_price) - 1.0
        fee_cost = half_fee + half_fee
        net = gross - fee_cost
        equity_val *= (1.0 + net)
        records.append({
            "entry_time": entry_time,
            "exit_time": exit_time,
            "side": "short",
            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_return": gross,
            "fee_cost": fee_cost,
            "net_return": net,
            "equity_after": equity_val,
        })
        pos = 0
        entry_price = None
        entry_time = None

    for _, row in df.iterrows():
        t = row[time_col]
        p = float(row[price_col])
        evt = row[event_col]

        if evt == "SELL":
            if pos == 0:
                pos = -1
                entry_price = p
                entry_time = t
            # if already short -> ignore
        else:  # BUY
            if pos == -1:
                close_short(t, p)
            # if flat -> ignore

    if close_on_last and pos == -1 and entry_price is not None:
        last_t = df.iloc[-1][time_col]
        last_p = float(df.iloc[-1][price_col])
        close_short(last_t, last_p)

    return _finish(records, initial_equity)


# --------------------------------------------------------------------
# Plot helper for cumulative return
# --------------------------------------------------------------------
def plot_cum_return(
    trades_df: pd.DataFrame,
    *,
    time_col: str = "exit_time",
    cumret_col: str = "cum_return",
    title: Optional[str] = "Cumulative Return",
):
    """
    Simple matplotlib plot of cumulative return (from trades_df).
    Expects a 'cum_return' column; if missing, computes it from 'net_return'.
    """
    import matplotlib.pyplot as plt

    if trades_df is None or trades_df.empty:
        raise ValueError("trades_df is empty")

    df = trades_df.copy()
    if cumret_col not in df.columns:
        if "net_return" not in df.columns:
            raise ValueError("trades_df missing 'cum_return' or 'net_return'")
        df[cumret_col] = (1.0 + df["net_return"]).cumprod() - 1.0

    x = pd.to_datetime(df[time_col], utc=True, errors="coerce")
    y = df[cumret_col].astype(float)

    plt.figure(figsize=(10, 4))
    plt.plot(x, y)
    plt.title(title if title else "")
    plt.xlabel("Time")
    plt.ylabel("Cumulative Return")
    plt.grid(True)
    plt.tight_layout()
    plt.show()



In [28]:
import numpy as np
import pandas as pd
from typing import Tuple, Optional, Dict

# ---------- utilities ----------
def _as_bool(s: pd.Series) -> pd.Series:
    return pd.Series(pd.to_numeric(s, errors="coerce").fillna(0).astype(bool), index=s.index)

def _bars_since_series(ev: pd.Series) -> pd.Series:
    ev = _as_bool(ev).astype(int).values
    n = len(ev)
    out = np.full(n, np.inf, dtype=float)
    last = -1
    for i in range(n):
        if ev[i] == 1:
            last = i
        out[i] = np.inf if last < 0 else (i - last)
    return pd.Series(out, index=ev.index)

def _kernel_changes(yhat1: pd.Series):
    y = pd.to_numeric(yhat1, errors="coerce")
    was_bull = y.shift(2) < y.shift(1)
    was_bear = y.shift(2) > y.shift(1)
    is_bull  = y.shift(1) < y
    is_bear  = y.shift(1) > y
    is_bullish_change  = (is_bull & was_bear).fillna(False)
    is_bearish_change  = (is_bear & was_bull).fillna(False)
    return is_bearish_change, is_bullish_change

def _finish_trade(trades, direction, entry_i, exit_i, prices, fee_frac, stop_loss_flag, exit_type):
    ep = prices.iloc[entry_i]
    xp = prices.iloc[exit_i]
    gross = (xp / ep - 1.0) if direction == "long" else (ep / xp - 1.0)
    net   = gross - fee_frac
    bars  = exit_i - entry_i
    trades.append({
        "direction": direction,
        "entry_time": prices.index[entry_i],
        "exit_time":  prices.index[exit_i],
        "entry_price": float(ep),
        "exit_price":  float(xp),
        "bars_held":   int(bars),
        "gross_return": float(gross),
        "net_return":   float(net),
        "stop_loss":    bool(stop_loss_flag),   # early flip happened before exit?
        "exit_type":    exit_type               # "strict" or "dynamic"
    })

def _stepwise_equity(trades_df: pd.DataFrame, time_index: pd.Index, start_value: float = 1.0) -> pd.Series:
    eq = pd.Series(start_value, index=time_index, dtype=float)
    cur = start_value
    for t, r in trades_df.set_index("exit_time")["net_return"].items():
        if t in eq.index:
            cur *= (1.0 + r)
            eq.loc[t:] = cur
    return eq

def _pick_col(df: pd.DataFrame, candidates):
    lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in df.columns:
            return name
        if name.lower() in lower:
            return lower[name.lower()]
    return None

def _add_cum_cols(trades_df: pd.DataFrame, start_value: float = 1.0) -> pd.DataFrame:
    if trades_df.empty:
        trades_df["cum_equity"] = []
        trades_df["cum_return"] = []
        return trades_df
    ce = []
    cur = start_value
    for r in trades_df["net_return"]:
        cur *= (1.0 + r)
        ce.append(cur)
    trades_df = trades_df.copy()
    trades_df["cum_equity"] = ce
    trades_df["cum_return"] = trades_df["cum_equity"] - 1.0
    return trades_df

def summarize_trades(trades_df: pd.DataFrame) -> Dict[str, float]:
    n = len(trades_df)
    wins = int((trades_df["net_return"] > 0).sum()) if n else 0
    losses = n - wins
    win_rate = wins / n if n else 0.0
    wl_ratio = (wins / max(losses, 1)) if n else 0.0
    stop_losses = int(trades_df["stop_loss"].sum()) if n else 0
    total_return = (trades_df["cum_return"].iloc[-1] if n else 0.0)
    return dict(
        total_trades=n,
        wins=wins,
        losses=losses,
        win_rate=win_rate,
        wl_ratio=wl_ratio,
        stop_losses=stop_losses,
        total_return=total_return,
    )

# ---------- LONG-ONLY (strict exits mimic Pine) ----------
def lc_backtest_long_only(
    out: pd.DataFrame,
    price_col: str = "close",
    hold_bars: int = 4,
    fee_frac: float = 0.002,
    use_dynamic_exits: bool = False,
    dynamic_is_valid: bool = False,
    yhat1_col: Optional[str] = None,
    start_equity: float = 1.0,
) -> Tuple[pd.DataFrame, pd.Series]:

    need = [price_col, "startLongTrade", "startShortTrade"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        raise ValueError(f"lc_backtest_long_only: missing columns {missing}")

    close = pd.to_numeric(out[price_col], errors="coerce")
    sL    = _as_bool(out["startLongTrade"])
    sS    = _as_bool(out["startShortTrade"])

    # dynamic exit (optional)
    if use_dynamic_exits and dynamic_is_valid:
        cand = yhat1_col or _pick_col(out, ["yhat1", "kernelEstimate", "kernel_estimate"])
        if not cand:
            raise ValueError("Dynamic exits requested, but no yhat1 column found. "
                             "Pass yhat1_col='...' or disable dynamic exits.")
        bear_change, _ = _kernel_changes(out[cand])
        bars_since_entry = _bars_since_series(sL)
        bars_since_exit  = _bars_since_series(bear_change)
        end_long_dynamic = (bear_change & (bars_since_exit > bars_since_entry).shift(1).fillna(False))
    else:
        end_long_dynamic = pd.Series(False, index=out.index)

    trades = []
    in_pos = False
    entry_i = None
    bars_in_trade = 0
    saw_early_flip = False  # record early flip but DO NOT exit early in strict mode

    for i in range(len(out)):
        # entry when flat
        if not in_pos and sL.iloc[i]:
            in_pos = True
            entry_i = i
            bars_in_trade = 0
            saw_early_flip = False
            continue

        if in_pos:
            bars_in_trade += 1

            # dynamic exit can close earlier
            if end_long_dynamic.iloc[i]:
                _finish_trade(trades, "long", entry_i, i, close, fee_frac, saw_early_flip, "dynamic")
                in_pos = False
                entry_i = None
                bars_in_trade = 0
                saw_early_flip = False
                continue

            # record early flip if opposite signal before 4 bars (Pine counts as stop-loss)
            if sS.iloc[i] and bars_in_trade < hold_bars:
                saw_early_flip = True
                # BUT keep holding to bar 4 (strict behavior)

            # strict exit exactly at bar 4
            if bars_in_trade >= hold_bars:
                _finish_trade(trades, "long", entry_i, i, close, fee_frac, saw_early_flip, "strict")
                in_pos = False
                entry_i = None
                bars_in_trade = 0
                saw_early_flip = False

    trades_df = _add_cum_cols(pd.DataFrame(trades), start_value=start_equity)
    equity = _stepwise_equity(trades_df, time_index=out.index, start_value=start_equity)
    return trades_df, equity

# ---------- SHORT-ONLY (strict exits mimic Pine) ----------
def lc_backtest_short_only(
    out: pd.DataFrame,
    price_col: str = "close",
    hold_bars: int = 4,
    fee_frac: float = 0.002,
    use_dynamic_exits: bool = False,
    dynamic_is_valid: bool = False,
    yhat1_col: Optional[str] = None,
    start_equity: float = 1.0,
) -> Tuple[pd.DataFrame, pd.Series]:

    need = [price_col, "startLongTrade", "startShortTrade"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        raise ValueError(f"lc_backtest_short_only: missing columns {missing}")

    close = pd.to_numeric(out[price_col], errors="coerce")
    sL    = _as_bool(out["startLongTrade"])
    sS    = _as_bool(out["startShortTrade"])

    if use_dynamic_exits and dynamic_is_valid:
        cand = yhat1_col or _pick_col(out, ["yhat1", "kernelEstimate", "kernel_estimate"])
        if not cand:
            raise ValueError("Dynamic exits requested, but no yhat1 column found.")
        _, bull_change = _kernel_changes(out[cand])
        bars_since_entry = _bars_since_series(sS)
        bars_since_exit  = _bars_since_series(bull_change)
        end_short_dynamic = (bull_change & (bars_since_exit > bars_since_entry).shift(1).fillna(False))
    else:
        end_short_dynamic = pd.Series(False, index=out.index)

    trades = []
    in_pos = False
    entry_i = None
    bars_in_trade = 0
    saw_early_flip = False

    for i in range(len(out)):
        if not in_pos and sS.iloc[i]:
            in_pos = True
            entry_i = i
            bars_in_trade = 0
            saw_early_flip = False
            continue

        if in_pos:
            bars_in_trade += 1

            if end_short_dynamic.iloc[i]:
                _finish_trade(trades, "short", entry_i, i, close, fee_frac, saw_early_flip, "dynamic")
                in_pos = False
                entry_i = None
                bars_in_trade = 0
                saw_early_flip = False
                continue

            if sL.iloc[i] and bars_in_trade < hold_bars:
                saw_early_flip = True  # record, but hold until bar 4

            if bars_in_trade >= hold_bars:
                _finish_trade(trades, "short", entry_i, i, close, fee_frac, saw_early_flip, "strict")
                in_pos = False
                entry_i = None
                bars_in_trade = 0
                saw_early_flip = False

    trades_df = _add_cum_cols(pd.DataFrame(trades), start_value=start_equity)
    equity = _stepwise_equity(trades_df, time_index=out.index, start_value=start_equity)
    return trades_df, equity

In [6]:
PRICE_CSV  = 'solusd_5m_coinbase.csv'
df_price  = pd.read_csv(PRICE_CSV, parse_dates=['ts'])

In [13]:
signals, out, lc = run_lorentzian_signals(
    df_price,                 # your OHLCV with either a DateTimeIndex or a 'ts' column
    tf="1h",
    start=df_price['date'][0],
    end="2025-09-30",
    neighbors_count=8,
    max_bars_back=None,     # None => full window
    color_compression=1.0,  # matches Pine default
    use_dynamic_exits=False,
    use_volatility_filter=True,
    use_regime_filter=True,
    use_adx_filter=False,
    regime_threshold=-0.1,
    adx_threshold=20
)

# See the first few events with strength + grade (mild/medium/strong)
signals.head()
# Save if you want:
signals.to_csv("lc_signals.csv", index=False)

In [29]:
# out = lc.df  # advanced-ta working frame; must contain close, startLongTrade, startShortTrade (and yhat1 if using dynamic exits)

# STRICT Pine-like exits (4 bars), no dynamic exits:
long_trades, long_eq   = lc_backtest_long_only(out, fee_frac=0.002, use_dynamic_exits=False, dynamic_is_valid=False)
short_trades, short_eq = lc_backtest_short_only(out, fee_frac=0.002, use_dynamic_exits=False, dynamic_is_valid=False)

# Quick stats and cumulative return:
print(summarize_trades(long_trades))
print(summarize_trades(short_trades))

# If you want earlier exits via kernel changes (only valid when EMA/SMA filters OFF and kernel smoothing OFF):
# long_trades_dyn, long_eq_dyn = lc_backtest_long_only(out, use_dynamic_exits=True, dynamic_is_valid=True, yhat1_col="yhat1")


{'total_trades': 599, 'wins': 249, 'losses': 350, 'win_rate': 0.41569282136894825, 'wl_ratio': 0.7114285714285714, 'stop_losses': 11, 'total_return': -0.7661361133474776}
{'total_trades': 600, 'wins': 240, 'losses': 360, 'win_rate': 0.4, 'wl_ratio': 0.6666666666666666, 'stop_losses': 20, 'total_return': -0.914736705338981}


In [27]:
np.sum(short_trades['bars_held']!=4)

20

In [23]:
short_trades

,direction,entry_time,exit_time,entry_price,exit_price,bars_held,gross_return,net_return
0,short,2021-07-01 04:00:00+00:00,2021-07-01 08:00:00+00:00,32.721,34.137,4,-0.041480,-0.043480
1,short,2021-07-03 22:00:00+00:00,2021-07-04 02:00:00+00:00,33.783,35.019,4,-0.035295,-0.037295
2,short,2021-07-04 20:00:00+00:00,2021-07-05 00:00:00+00:00,34.207,33.595,4,0.018217,0.016217
3,short,2021-07-07 21:00:00+00:00,2021-07-08 01:00:00+00:00,36.417,36.779,4,-0.009843,-0.011843
4,short,2021-07-10 08:00:00+00:00,2021-07-10 12:00:00+00:00,32.676,32.410,4,0.008207,0.006207
...,...,...,...,...,...,...,...,...
595,short,2025-09-14 11:00:00+00:00,2025-09-14 15:00:00+00:00,242.580,243.940,4,-0.005575,-0.007575
596,short,2025-09-19 00:00:00+00:00,2025-09-19 04:00:00+00:00,244.280,244.190,4,0.000369,-0.001631
597,short,2025-09-21 06:00:00+00:00,2025-09-21 10:00:00+00:00,238.710,239.620,4,-0.003798,-0.005798
598,short,2025-09-24 22:00:00+00:00,2025-09-25 02:00:00+00:00,209.200,205.240,4,0.019294,0.017294
